In [4]:
# Verificación de disponibilidad de la GPU en Google Colab
!nvidia-smi

Sat Oct 10 17:42:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install transformers datasets evaluate accelerate scikit-learn pandas torch huggingface_hub

In [6]:
import os
import time
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import pipeline
from huggingface_hub import login

# Configurar dispositivo GPU (device=0 para PyTorch en CUDA)
device = 0 if torch.cuda.is_available() else -1
print(f"Dispositivo de ejecución asignado: {'GPU (CUDA)' if device == 0 else 'CPU'}")

# Autenticación interactiva o por token de entorno
# os.environ["HF_TOKEN"] = "tu_token_aqui"
# login(token=os.environ.get("HF_TOKEN"))

Dispositivo de ejecución asignado: GPU (CUDA)


In [7]:
# Carga del dataset de opiniones de restaurantes
data = {
    "comentario": [
        "El servicio fue excelente y la comida llegó rapidísimo. Muy recomendado.",
        "La comida tardó más de 45 minutos y llegó fría. Deplorable atención.",
        "El lugar es agradable, pero los precios son algo elevados para la porción.",
        "Pésimo trato del mesero, nos ignoró durante media hora.",
        "Las hamburguesas tienen muy buen sabor, volvería sin duda alguna.",
        "Atención regular, nada fuera de lo común. La limonada estaba tibia.",
        "Me encantó el postre de chocolate, excelente presentación.",
        "La mesa estaba sucia al llegar y tardaron en limpiarla.",
        "Buena relación calidad-precio en el menú del día.",
        "No me gustó la salsa, estaba demasiado salada y el aire acondicionado no servía."
    ],
    "sentimiento_real": ["POS", "NEG", "NEU", "NEG", "POS", "NEU", "POS", "NEG", "POS", "NEG"]
}

df = pd.DataFrame(data)
print(f"Total de registros cargados: {len(df)}")
df.head()

Total de registros cargados: 10


,comentario,sentimiento_real
0,El servicio fue excelente y la comida llegó ra...,POS
1,La comida tardó más de 45 minutos y llegó fría...,NEG
2,"El lugar es agradable, pero los precios son al...",NEU
3,"Pésimo trato del mesero, nos ignoró durante me...",NEG
4,"Las hamburguesas tienen muy buen sabor, volver...",POS


In [8]:
# Carga del dataset de opiniones de restaurantes
data = {
    "comentario": [
        "El servicio fue excelente y la comida llegó rapidísimo. Muy recomendado.",
        "La comida tardó más de 45 minutos y llegó fría. Deplorable atención.",
        "El lugar es agradable, pero los precios son algo elevados para la porción.",
        "Pésimo trato del mesero, nos ignoró durante media hora.",
        "Las hamburguesas tienen muy buen sabor, volvería sin duda alguna.",
        "Atención regular, nada fuera de lo común. La limonada estaba tibia.",
        "Me encantó el postre de chocolate, excelente presentación.",
        "La mesa estaba sucia al llegar y tardaron en limpiarla.",
        "Buena relación calidad-precio en el menú del día.",
        "No me gustó la salsa, estaba demasiado salada y el aire acondicionado no servía."
    ],
    "sentimiento_real": ["POS", "NEG", "NEU", "NEG", "POS", "NEU", "POS", "NEG", "POS", "NEG"]
}

df = pd.DataFrame(data)
print(f"Total de registros cargados: {len(df)}")
df.head()

Total de registros cargados: 10


,comentario,sentimiento_real
0,El servicio fue excelente y la comida llegó ra...,POS
1,La comida tardó más de 45 minutos y llegó fría...,NEG
2,"El lugar es agradable, pero los precios son al...",NEU
3,"Pésimo trato del mesero, nos ignoró durante me...",NEG
4,"Las hamburguesas tienen muy buen sabor, volver...",POS


In [9]:
# Definición de los 3 modelos a evaluar del Hub de Hugging Face
models_info = [
    {
        "id": "pysentimiento/robertuito-sentiment-analysis",
        "name": "RoBERTuito (Español)",
        "mapping": {"POS": "POS", "NEG": "NEG", "NEU": "NEU"}
    },
    {
        "id": "finiteautomata/beto-sentiment-analysis",
        "name": "BETO Sentiment (Español)",
        "mapping": {"POS": "POS", "NEG": "NEG", "NEU": "NEU"}
    },
    {
        "id": "cardiffnlp/twitter-xlm-roberta-base-sentiment",
        "name": "XLM-RoBERTa Multilingüe",
        "mapping": {"positive": "POS", "negative": "NEG", "neutral": "NEU", "POS": "POS", "NEG": "NEG", "NEU": "NEU"}
    }
]

benchmark_results = []

for m_info in models_info:
    model_id = m_info["id"]
    model_name = m_info["name"]
    mapping = m_info["mapping"]

    print(f"\n==========================================")
    print(f"Cargando e infiriendo modelo: {model_name}")
    print(f"==========================================")

    # Inicialización de pipeline en GPU
    classifier = pipeline(
        task="text-classification",
        model=model_id,
        tokenizer=model_id,
        device=device
    )

    # Medición de latencia de inferencia
    start_time = time.time()
    raw_predictions = classifier(df["comentario"].tolist())
    total_time = time.time() - start_time
    avg_latency_ms = (total_time / len(df)) * 1000

    # Mapeo y normalización de etiquetas predecidas
    preds_normalized = []
    for pred in raw_predictions:
        raw_label = pred["label"].upper() if pred["label"] in ["POS", "NEG", "NEU"] else pred["label"].lower()
        norm_label = mapping.get(raw_label, mapping.get(pred["label"], "NEU"))
        preds_normalized.append(norm_label)

    # Cálculo de métricas estándar
    y_true = df["sentimiento_real"].tolist()
    acc = accuracy_score(y_true, preds_normalized)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, preds_normalized, average="weighted", zero_division=0)

    benchmark_results.append({
        "Modelo": model_name,
        "HuggingFace ID": model_id,
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Latencia Total (s)": round(total_time, 4),
        "Latencia Promedio (ms)": round(avg_latency_ms, 2)
    })

df_results = pd.DataFrame(benchmark_results)


Cargando e infiriendo modelo: RoBERTuito (Español)


config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  435MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [10]:
# Despliegue de tabla técnica comparativa de resultados
print("\n--- RESUMEN COMPARATIVO DE MODELOS ---")
display(df_results)


--- RESUMEN COMPARATIVO DE MODELOS ---


,Modelo,HuggingFace ID,Accuracy,Precision,Recall,F1-Score,Latencia Total (s),Latencia Promedio (ms)
0,RoBERTuito (Español),pysentimiento/robertuito-sentiment-analysis,1.0,1.0000,1.0,1.00,1.9486,194.86
1,BETO Sentiment (Español),finiteautomata/beto-sentiment-analysis,1.0,1.0000,1.0,1.00,0.5554,55.54
2,XLM-RoBERTa Multilingüe,cardiffnlp/twitter-xlm-roberta-base-sentiment,0.8,0.6667,0.8,0.72,0.2217,22.17


In [11]:
print("--- CONCLUSIONES PARCIALES ---")
best_model = df_results.sort_values(by="F1-Score", ascending=False).iloc[0]
print(f"1. El modelo con mejor desempeño global es '{best_model['Modelo']}' con un F1-Score de {best_model['F1-Score']}.")
print(f"2. Latencia promedio por inferencia en GPU T4: {best_model['Latencia Promedio (ms)']} ms.")
print(f"3. La normalización de etiquetas fue requerida para homogenizar las salidas de etiquetas categóricas entre arquitecturas.")

--- CONCLUSIONES PARCIALES ---
1. El modelo con mejor desempeño global es 'RoBERTuito (Español)' con un F1-Score de 1.0.
2. Latencia promedio por inferencia en GPU T4: 194.86 ms.
3. La normalización de etiquetas fue requerida para homogenizar las salidas de etiquetas categóricas entre arquitecturas.
